# FH-06 · Final report for the CEO
Collects everything saved in `results/` into: (1) run comparison, (2) acceptance table against the **proposed** thresholds, (3) one chart, (4) a plain-language summary.
No GPU needed. Re-run after each stage; change `TAGS` to compare base vs trained adapters.

In [ ]:
# ---- Colab setup: upload futurehuman_testing_suite.zip when asked (once per session) ----
import os, sys, subprocess
SUITE = "/content/futurehuman_testing_suite"
if not os.path.isdir(SUITE):
    from google.colab import files
    up = files.upload()
    subprocess.run(["unzip", "-o", "-q", list(up)[0], "-d", "/content"], check=True)
os.chdir(SUITE); sys.path.insert(0, SUITE)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "accelerate", "bitsandbytes", "tabulate"])
print("ready in", os.getcwd())

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
from fh.report import load_runs, comparison_table, load_capabilities, acceptance_table, markdown_report
TAGS = ["base"]            # e.g. ["base", "stage1", "stage2"]
runs = load_runs("results", split="test")
print("runs:", list(runs))
comparison_table(runs)

In [ ]:
for t in TAGS:
    caps = load_capabilities(t)
    print("=== acceptance table for", t, "(thresholds are PROPOSED - business owner decides) ===")
    print(acceptance_table(caps).to_string(index=False))

In [ ]:
from fh.metrics import summarize, risk_coverage
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
names, far, f1 = [], [], []
for k, d in runs.items():
    s = summarize(d); names.append(k.replace("base_", "").replace("_test", "")); far.append(s["safety"]["false_action"]["rate"] * 100); f1.append(s["macro_f1"])
ax[0].bar(names, far, color="#c0392b"); ax[0].set_ylabel("false-action rate (%)  - lower is better"); ax[0].tick_params(axis="x", rotation=30); ax[0].set_title("Headline safety number")
ax[1].bar(names, f1, color="#2e86c1"); ax[1].set_ylabel("macro-F1 (actionability)"); ax[1].tick_params(axis="x", rotation=30); ax[1].set_title("Overall quality")
plt.tight_layout(); plt.savefig("results/ceo_chart.png", dpi=160); plt.show()

In [ ]:
caps = load_capabilities(TAGS[0]); v = lambda k, f="{:.1%}": (f.format(caps[k]["value"]) if k in caps else "not measured")
print(f'''
WHAT WE TESTED
We tested the model on {len(runs[list(runs)[0]])} held-out conversations (English, Hinglish, Hindi): does it act when it should, stay silent when it should, and ask when unsure.
The model only proposes; a separate rule-based gate decides, and all tools are simulated.

HEADLINE NUMBERS
- False-action rate (acted when it should not have): {v('false_action_rate')}
- False-execution rate after the gate (wrong change to calendar/messages): {v('false_execution_rate')}
- Task completion on simulated tools: {v('task_completion')}
- Valid structured output: {v('schema_valid_rate')}

WHAT IS NOT TESTED YET
- Trained adapters (Stage 1 / Stage 2): the harness is ready; rerun with TAG and ADAPTER.
- Real audio in real rooms, a test set of 300+ conversations with several independent annotators, real calendar/message APIs, DPO.
''')

In [ ]:
open("results/REPORT.md", "w").write(markdown_report(runs))
import shutil; shutil.make_archive("/content/fh_results", "zip", "results")
from google.colab import files; files.download("/content/fh_results.zip")